# Notebook 2b: Model Training with Binary eDNA Presence/Absence

---

**Author**: Laura Steel Pascual  
**Date**: February 2026  
**Project**: Decoding Environmental Status with eDNA and Supervised Machine Learning: Insights into Ecological Condition and Species‑Level Predictors in Solomon Island Rivers

---

## Overview
This notebook trains and evaluates supervised learning classifiers to predict site environmental status (reference vs impacted) from binary eDNA species presence/absence encoding rather than read count data. Decision: Count data retained for all subsequent modelling.

**Purpose**: 
- Train 4 classifier types across 8 buffer-threshold combinations using binary eDNA data.
- Compare binary vs count-based model performance to quantify the classification-interpretability trade-off.

**Methods:**
- Classifiers: Decision Tree, Random Forest, XGBoost, Logistic Regression.
- Pipeline: binary conversion (counts 0/1) -> classification.
- Cross-validation: Stratified 5-fold.
- Metrics: AUC, Accuracy, Precision, Recall, F1.

**Paper Reference**: 2. Materials & Methods -> 2.3 Data Processing and Preprocessing

---

## 1. Setup

In [3]:
# Import libraries
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
import pickle

# Create output directories
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/model_performance', exist_ok=True)

print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'Xgboost version: {xgboost.__version__}')
print(f'Matplotlib version: {matplotlib.__version__}')
print(f'Seaborn version: {sns.__version__}')

NumPy version: 1.24.4
Pandas version: 2.3.2
Scikit-learn version: 1.2.2
Xgboost version: 3.0.3
Matplotlib version: 3.9.2
Seaborn version: 0.12.2


## 2. Load Preprocessed Data and Binary Conversion

Load raw count data from Notebook 1, convert to binary presence/absence.

In [5]:
buffers = [100, 300, 500, 1000]
thresholds = [0.03125, 0.0625]  # ← CORRECTED: was [0.003125, 0.00625]

datasets = {}
for buf in buffers:
    for thresh in thresholds:
        thresh_str = str(thresh).replace('.', '')  # ← CORRECTED: single quotes & dot
        filename = f'data_{buf}m_t{thresh_str}.pkl'  # ← CORRECTED: single quotes
        filepath = os.path.join('results/preprocessed', filename)  # ← CORRECTED: single quotes
        
        with open(filepath, 'rb') as f:  # ← CORRECTED: single quotes
            data = pickle.load(f)
        
        key = f'{buf}m_t{thresh_str}'  # ← CORRECTED: single quotes
        datasets[key] = data  # ← Keep original structure first
        
        # Convert raw counts to binary (1=present, 0=absent)
        X_raw = data["X_raw"]
        X_binary = (X_raw > 0).astype(int)
        
        # Store binary version
        datasets[key] = {
            "X_binary": X_binary,
            "y": data["y"],
            "feature_names": data["feature_names"],
            "n_reference": np.sum(data["y"] == 0),
            "n_impacted": np.sum(data["y"] == 1)
        }
        
        print(f'{key}: {datasets[key]["n_reference"]} ref, {datasets[key]["n_impacted"]} imp, {X_binary.shape[1]} binary features')

print(f'\nLoaded {len(datasets)} binary datasets')
print(f'Features per dataset: {len(datasets["100m_t003125"]["feature_names"])}')


100m_t003125: 26 ref, 21 imp, 861 binary features
100m_t00625: 27 ref, 20 imp, 861 binary features
300m_t003125: 22 ref, 25 imp, 861 binary features
300m_t00625: 25 ref, 22 imp, 861 binary features
500m_t003125: 16 ref, 31 imp, 861 binary features
500m_t00625: 23 ref, 24 imp, 861 binary features
1000m_t003125: 18 ref, 29 imp, 861 binary features
1000m_t00625: 29 ref, 18 imp, 861 binary features

Loaded 8 binary datasets
Features per dataset: 861


### 2.1 Quick X_binary Inspection

In [7]:
# Pick first dataset for inspection
example_key = "100m_t003125"
X_bin = datasets[example_key]["X_binary"]
y = datasets[example_key]["y"]

print(f"Example dataset: {example_key}")
print(f"Shape: {X_bin.shape} (n_samples={X_bin.shape[0]}, n_features={X_bin.shape[1]})")
print(f"Data type: {X_bin.dtype}")
print(f"Memory usage: {X_bin.nbytes / 1e6:.1f} MB")

print(f"\nFirst 5 rows, first 10 columns:")
print(pd.DataFrame(X_bin[:, :10], columns=datasets[example_key]["feature_names"][:10]).head())

print(f"\nPresence summary:")
print(f"  Total 1's:  {X_bin.sum():>6,} ({100*X_bin.sum()/X_bin.size:.1f}%)")
print(f"  Total 0's: {X_bin.size - X_bin.sum():>6,} ({100*(1-X_bin.sum()/X_bin.size):.1f}%)")
print(f"  Features with ANY presence: {100*(X_bin.sum(0)>0).sum()/X_bin.shape[1]:.1f}%")
print(f"  Features ALWAYS absent:    {(X_bin.sum(0)==0).sum():>3} ({100*(X_bin.sum(0)==0).sum()/X_bin.shape[1]:.1f}%)")

# Check if truly binary
unique_vals = np.unique(X_bin)
print(f"\nUnique values: {unique_vals}")
assert set(unique_vals).issubset({0,1}), f"Non-binary values found: {set(unique_vals)}"

# Per-class presence
print(f"\nReference sites (n={np.sum(y==0)}): {100*X_bin[y==0].sum()/(X_bin[y==0].size):.1f}% presence")
print(f"Impacted sites (n={np.sum(y==1)}):  {100*X_bin[y==1].sum()/(X_bin[y==1].size):.1f}% presence")

print("\nBinary conversion verified - ready for modelling")


Example dataset: 100m_t003125
Shape: (47, 861) (n_samples=47, n_features=861)
Data type: int32
Memory usage: 0.2 MB

First 5 rows, first 10 columns:
   Sicyopterus cynocephalus or lagocephalus  Areca catechu  \
0                                         1              1   
1                                         1              1   
2                                         1              1   
3                                         1              1   
4                                         1              1   

   Rhyacichthys guilberti  Stiphodon semoni  Kleinhovia hospita  \
0                       1                 1                   1   
1                       1                 1                   0   
2                       1                 1                   1   
3                       1                 1                   1   
4                       1                 1                   1   

   Kuhlia marginata  Ludwigia sp. C-006  Crenimugil cf. heterocheilos  \
0 

## 3. Define ML Pipeline (Binary)

Define 4 classifiers with minimal pipeline. No preprocessing needed for binary 0/1 data (tree-based models scale-invariant; logistic regression performs well on uniform-scale features).

In [9]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

classifiers = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1),
    "Logistic Regression": LogisticRegression(random_state=42, n_jobs=-1)
}

print("Classifiers:", list(classifiers.keys()))
print("CV: 5-fold stratified")

Classifiers: ['Decision Tree', 'Random Forest', 'XGBoost', 'Logistic Regression']
CV: 5-fold stratified


## 4. Classifier Benchmarking (5-Fold CV)

Benchmark all 4 classifiers across 8 buffer-threshold combinations using identical 5-fold CV as count-based Notebook 2.

In [11]:
results_5fold = {}

for dataset_key, data in datasets.items():
    print(f"\n{'='*80}")
    print(f"Dataset: {dataset_key}")
    print('='*80)
    
    X_binary = data["X_binary"]
    y = data["y"]
    
    results_5fold[dataset_key] = {}
    
    for clf_name, clf in classifiers.items():
        print(f"\n{clf_name}")
        
        # Binary pipeline: JUST classifier (no scaler/log needed)
        pipe = Pipeline([("clf", clf)])
        
        preds = np.zeros_like(y, dtype=int)
        acc_scores, auc_scores = [], []
        train_acc_scores, train_auc_scores = [], []
        
        for train_idx, test_idx in cv_strategy.split(X_binary, y):
            X_train, X_test = X_binary[train_idx], X_binary[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)
            preds[test_idx] = y_pred
            y_train_pred = pipe.predict(X_train)
            
            acc_scores.append(accuracy_score(y_test, y_pred))
            train_acc_scores.append(accuracy_score(y_train, y_train_pred))
            
            # AUC (proba or decision function)
            try:
                if hasattr(pipe.named_steps["clf"], "predict_proba"):
                    y_proba = pipe.predict_proba(X_test)[:, 1]
                    y_train_proba = pipe.predict_proba(X_train)[:, 1]
                else:
                    y_proba = pipe.decision_function(X_test)
                    y_train_proba = pipe.decision_function(X_train)
                
                auc_scores.append(roc_auc_score(y_test, y_proba))
                train_auc_scores.append(roc_auc_score(y_train, y_train_proba))
            except:
                auc_scores.append(np.nan)
                train_auc_scores.append(np.nan)
        
        print(f"Train Acc:  {np.mean(train_acc_scores):.4f} ± {np.std(train_acc_scores):.4f}")
        print(f"Train AUC:  {np.nanmean(train_auc_scores):.4f} ± {np.nanstd(train_auc_scores):.4f}")
        print(f"Val Acc:    {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
        print(f"Val AUC:    {np.nanmean(auc_scores):.4f} ± {np.nanstd(auc_scores):.4f}")
        print("\nClassification report (aggregated CV predictions):")
        print(classification_report(y, preds))
        
        results_5fold[dataset_key][clf_name] = {
            "train_accuracy": train_acc_scores,
            "test_accuracy": acc_scores,
            "train_auc": train_auc_scores,
            "test_auc": auc_scores,
            "predictions": preds
        }

print("\n Binary 5-fold CV benchmarking complete")



Dataset: 100m_t003125

Decision Tree
Train Acc:  1.0000 ± 0.0000
Train AUC:  1.0000 ± 0.0000
Val Acc:    0.7000 ± 0.1322
Val AUC:    0.6967 ± 0.1425

Classification report (aggregated CV predictions):
              precision    recall  f1-score   support

           0       0.73      0.73      0.73        26
           1       0.67      0.67      0.67        21

    accuracy                           0.70        47
   macro avg       0.70      0.70      0.70        47
weighted avg       0.70      0.70      0.70        47


Random Forest
Train Acc:  1.0000 ± 0.0000
Train AUC:  1.0000 ± 0.0000
Val Acc:    0.5711 ± 0.1405
Val AUC:    0.5710 ± 0.1642

Classification report (aggregated CV predictions):
              precision    recall  f1-score   support

           0       0.59      0.77      0.67        26
           1       0.54      0.33      0.41        21

    accuracy                           0.57        47
   macro avg       0.56      0.55      0.54        47
weighted avg       0

### 4.1 Performance Summary and Export

Compile performance summary table for easy comparison between binary vs count representations.

In [13]:
summary_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key to extract buffer and threshold
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    for clf_name in classifiers.keys():
        result = results_5fold[dataset_key][clf_name]
        
        summary_rows.append({
            'Buffer (m)': buffer,
            'Threshold': threshold,
            'Dataset': dataset_key,
            'Classifier': clf_name,
            'Train Accuracy': np.mean(result['train_accuracy']),
            'Train Accuracy Std': np.std(result['train_accuracy']),
            'Test Accuracy': np.mean(result['test_accuracy']),
            'Test Accuracy Std': np.std(result['test_accuracy']),
            'Train AUC': np.nanmean(result['train_auc']),
            'Train AUC Std': np.nanstd(result['train_auc']),
            'Test AUC': np.nanmean(result['test_auc']),
            'Test AUC Std': np.nanstd(result['test_auc'])
        })

summary_df = pd.DataFrame(summary_rows)

# Display test performance only
print("\n5-Fold CV Performance Summary (Test Set):")
print("="*90)
test_cols = ['Buffer (m)', 'Threshold', 'Classifier', 'Test Accuracy', 'Test Accuracy Std', 
             'Test AUC', 'Test AUC Std']
print(summary_df[test_cols].to_string(index=False))

# Save full table to Excel (includes both train and test)
summary_df.to_excel('results/model_performance/binary_5fold_cv_summary.xlsx', index=False)
print("\n" + "="*90)
print("Saved: results/model_performance/binary_5fold_cv_summary.xlsx")



5-Fold CV Performance Summary (Test Set):
 Buffer (m)  Threshold          Classifier  Test Accuracy  Test Accuracy Std  Test AUC  Test AUC Std
        100   0.003125       Decision Tree       0.700000           0.132218  0.696667      0.142459
        100   0.003125       Random Forest       0.571111           0.140475  0.571000      0.164207
        100   0.003125             XGBoost       0.511111           0.159629  0.542333      0.176451
        100   0.003125 Logistic Regression       0.660000           0.118342  0.688333      0.118509
        100   0.006250       Decision Tree       0.595556           0.165939  0.591667      0.153297
        100   0.006250       Random Forest       0.568889           0.170605  0.616667      0.186488
        100   0.006250             XGBoost       0.573333           0.176887  0.580000      0.140000
        100   0.006250 Logistic Regression       0.637778           0.143604  0.726667      0.109214
        300   0.003125       Decision Tree      

### 4.2 Class-wise Metrics
Extract Precision/Recall/F1 per class (Reference vs Impacted) for balanced performance assessment.

In [15]:
classwise_rows = []

for dataset_key in datasets.keys():
    # Parse dataset key
    parts = dataset_key.split('_')
    buffer = int(parts[0].replace('m', ''))
    threshold_str = parts[1].replace('t', '')
    threshold = float('0.' + threshold_str)
    
    for clf_name in classifiers.keys():
        result = results_5fold[dataset_key][clf_name]
        
        # Get predictions
        y = datasets[dataset_key]['y']
        preds = result['predictions']
        
        # Calculate class-wise metrics
        from sklearn.metrics import precision_recall_fscore_support
        precision, recall, f1, support = precision_recall_fscore_support(
            y, preds, average=None, zero_division=0
        )
        
        # Add rows for each class
        for class_idx in [0, 1]:
            class_label = 'Reference' if class_idx == 0 else 'Impacted'
            
            classwise_rows.append({
                'Buffer (m)': buffer,
                'Threshold': threshold,
                'Dataset': dataset_key,
                'Classifier': clf_name,
                'Class': class_idx,
                'Class_Label': class_label,
                'Precision': precision[class_idx],
                'Recall': recall[class_idx],
                'F1-Score': f1[class_idx],
                'Support': support[class_idx],
                'Mean_Test_Accuracy': np.mean(result['test_accuracy']),
                'Mean_Test_AUC': np.nanmean(result['test_auc'])
            })

classwise_df = pd.DataFrame(classwise_rows)

# Display format (matching original exactly)
print("\n" + "="*100)
print("Class-wise Performance Metrics (5-Fold CV):")
print("="*100)

for dataset_key in datasets.keys():
    print(f"\n{dataset_key}:")
    print("-" * 90)
    
    dataset_data = classwise_df[classwise_df['Dataset'] == dataset_key]
    
    for clf_name in classifiers.keys():
        clf_data = dataset_data[dataset_data['Classifier'] == clf_name]
        if len(clf_data) > 0:
            print(f"\n  {clf_name}:")
            print(f"    {'Class':<6} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support':<10}")
            print(f"    {'-'*52}")
            for _, row in clf_data.iterrows():
                print(f"    {row['Class']:<6} {row['Precision']:<12.2f} "
                      f"{row['Recall']:<12.2f} {row['F1-Score']:<12.2f} {row['Support']:<10.0f}")
            
            # Print mean metrics
            mean_acc = clf_data.iloc[0]['Mean_Test_Accuracy']
            mean_auc = clf_data.iloc[0]['Mean_Test_AUC']
            print(f"    Mean Test Accuracy: {mean_acc:.4f}")
            print(f"    Mean Test AUC: {mean_auc:.4f}")

# Save to Excel
classwise_df.to_excel('results/model_performance/binary_5fold_cv_classwise_metrics.xlsx', index=False)
print("\n\nSaved: results/model_performance/binary_5fold_cv_classwise_metrics.xlsx")



Class-wise Performance Metrics (5-Fold CV):

100m_t003125:
------------------------------------------------------------------------------------------

  Decision Tree:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.73         0.73         0.73         26        
    1      0.67         0.67         0.67         21        
    Mean Test Accuracy: 0.7000
    Mean Test AUC: 0.6967

  Random Forest:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.59         0.77         0.67         26        
    1      0.54         0.33         0.41         21        
    Mean Test Accuracy: 0.5711
    Mean Test AUC: 0.5710

  XGBoost:
    Class  Precision    Recall       F1           Support   
    ----------------------------------------------------
    0      0.56         0.58         0.57         26        
    1      0.45         0.43

### 4.3 Variance Analysis
Quantify CV stability.

In [17]:
# Extract variance metrics for comparison
variance_comparison = []

for dataset_key in datasets.keys():
    for clf_name in classifiers.keys():
        result = results_5fold[dataset_key][clf_name]
        
        variance_comparison.append({
            'Dataset': dataset_key,
            'Classifier': clf_name,
            'CV_Folds': 5,
            'Acc_Mean': np.mean(result['test_accuracy']),
            'Acc_Std': np.std(result['test_accuracy']),
            'AUC_Mean': np.nanmean(result['test_auc']),
            'AUC_Std': np.nanstd(result['test_auc'])
        })

variance_df = pd.DataFrame(variance_comparison)

# Summary statistics
print("\n" + "="*90)
print("Variance Analysis: 5-Fold CV Stability")
print("="*90)
print(f"\nMean AUC Std across all configurations: {variance_df['AUC_Std'].mean():.4f}")
print(f"Mean Accuracy Std across all configurations: {variance_df['Acc_Std'].mean():.4f}")
print(f"\nConfigurations with AUC Std > 0.20: {(variance_df['AUC_Std'] > 0.20).sum()}/{len(variance_df)}")
print(f"Configurations with Accuracy Std > 0.20: {(variance_df['Acc_Std'] > 0.20).sum()}/{len(variance_df)}")

# Save to Excel
variance_df.to_excel('results/model_performance/binary_5fold_variance_analysis.xlsx', index=False)
print("\nSaved: results/model_performance/binary_5fold_variance_analysis.xlsx")


Variance Analysis: 5-Fold CV Stability

Mean AUC Std across all configurations: 0.1123
Mean Accuracy Std across all configurations: 0.1044

Configurations with AUC Std > 0.20: 0/32
Configurations with Accuracy Std > 0.20: 0/32

Saved: results/model_performance/binary_5fold_variance_analysis.xlsx


### 4.4 Direct Comparison: Binary vs Count-Based Models

Directly compare binary (presence/absence) vs count-based model performance to quantify the classification-interpretability trade-off. Binary models may achieve superior accuracy but cannot reveal how read count magnitudes influence predictions—essential for SHAP analysis of taxon-specific dose-response effects.

In [19]:
# Load count-based results from Notebook 2
count_summary = pd.read_excel('results/model_performance/5fold_cv_summary.xlsx')

# Prepare binary results (already in memory)
binary_summary = summary_df.copy()

# Add representation labels
count_summary['Representation'] = 'Count'
binary_summary['Representation'] = 'Binary'

# Combine datasets
comparison_df = pd.concat([count_summary, binary_summary], ignore_index=True)

print("\n" + "="*100)
print("Binary vs Count-Based Performance Comparison")
print("="*100)

# Direct comparison by configuration
comparison_rows = []

for (buf, thresh, clf), group in comparison_df.groupby(['Buffer (m)', 'Threshold', 'Classifier']):
    count_row = group[group['Representation'] == 'Count'].iloc[0]
    binary_row = group[group['Representation'] == 'Binary'].iloc[0]
    
    auc_diff = binary_row['Test AUC'] - count_row['Test AUC']
    acc_diff = binary_row['Test Accuracy'] - count_row['Test Accuracy']
    
    comparison_rows.append({
        'Buffer (m)': buf,
        'Threshold': thresh,
        'Classifier': clf,
        'Count AUC': count_row['Test AUC'],
        'Binary AUC': binary_row['Test AUC'],
        'AUC Diff (Binary - Count)': auc_diff,
        'Count Acc': count_row['Test Accuracy'],
        'Binary Acc': binary_row['Test Accuracy'],
        'Acc Diff (Binary - Count)': acc_diff,
        'Binary Superior (AUC)': auc_diff > 0,
        'Binary Superior (Acc)': acc_diff > 0
    })

comp_df = pd.DataFrame(comparison_rows)

# Overall statistics
print(f"\nOverall Statistics (32 configurations = 8 buffer-threshold combos × 4 classifiers):")
print(f"  Binary superior in AUC:      {comp_df['Binary Superior (AUC)'].sum()}/{len(comp_df)} ({100*comp_df['Binary Superior (AUC)'].mean():.1f}%)")
print(f"  Binary superior in Accuracy: {comp_df['Binary Superior (Acc)'].sum()}/{len(comp_df)} ({100*comp_df['Binary Superior (Acc)'].mean():.1f}%)")
print(f"\n  Mean AUC difference (Binary - Count):      {comp_df['AUC Diff (Binary - Count)'].mean():+.4f}")
print(f"  Mean Accuracy difference (Binary - Count): {comp_df['Acc Diff (Binary - Count)'].mean():+.4f}")


Binary vs Count-Based Performance Comparison

Overall Statistics (32 configurations = 8 buffer-threshold combos × 4 classifiers):
  Binary superior in AUC:      14/32 (43.8%)
  Binary superior in Accuracy: 12/32 (37.5%)

  Mean AUC difference (Binary - Count):      -0.0171
  Mean Accuracy difference (Binary - Count): -0.0125


## 5. Summary

**Outputs** (`results/model_performance/`):
- `binary_5fold_cv_summary.xlsx` 
- `binary_5fold_cv_classwise_metrics.xlsx` 
- `binary_5fold_variance_analysis.xlsx` 

**Notable result:** At 1000m / 0.0625, binary Random Forest achieved the study's highest AUC of 0.990 (± 0.017), outperforming the count-based equivalent (AUC = 0.939 ± 0.042), suggesting species composition alone may suffice for site-level prediction at broad spatial scales.

**Decision: Count data retained for all subsequent modelling.** Binary encoding cannot reveal how read count magnitude influences predictions — whether species exert effects through detection-strength gradients or through presence alone. Count-based models are required for the SHAP dose-response interpretation in Notebook 4. The marginal mean performance advantage (0.017 AUC) is secondary to this mechanistic interpretability requirement.

## Next Steps

Return to Notebook 2 (`2_Model_Training.ipynb`) for hyperparameter-tuned count-based models.